# Dados IBGE

Municipal book of territory, Census population and SIDRA GDP.
Variable definitions are in `dictionary.csv`.
Raw extracts stay in `data/` and are not versioned.


## Imports

## Base IBGE

In [ ]:
import math
import os
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode


## Configurações

In [ ]:
def resolve_repo_root() -> Path:
    """Locate the project root that contains the local data directory."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "books").is_dir():
            return candidate
    return start


BASE_DIR = resolve_repo_root() / "data"
RAW_DIR = BASE_DIR / "raw" / "ibge"
TRUSTED_DIR = BASE_DIR / "trusted" / "ibge"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## Funções

In [ ]:
def get_json(url: str, params: Optional[dict] = None, timeout: int = 60) -> Union[list, dict]:
    """
    Perform GET request and return parsed JSON (list or dict).
    """
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.json()


## Cadastro de municípios do IBGE

In [ ]:
def baixar_municipios_ibge() -> pd.DataFrame:
    """
    Baixa todos os municípios brasileiros pela API de Localidades do IBGE.
    Retorna uma tabela padronizada com código IBGE, município, UF e região.
    """
    url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    data = get_json(url)

    rows = []

    for item in data:
        # IBGE may return null for nested objects; .get(key, {}) still yields None if key exists with null.
        microrregiao = item.get("microrregiao") or {}
        mesorregiao = microrregiao.get("mesorregiao") or {}
        uf = mesorregiao.get("UF") or {}
        regiao = uf.get("regiao") or {}

        regiao_imediata = item.get("regiao-imediata") or {}
        regiao_intermediaria = regiao_imediata.get("regiao-intermediaria") or {}

        rows.append({
            "codigo_ibge": str(item.get("id")).zfill(7),
            "municipio": item.get("nome"),
            "municipio_norm": unidecode(item.get("nome", "")).upper(),
            "uf": uf.get("sigla"),
            "codigo_uf": uf.get("id"),
            "nome_uf": uf.get("nome"),
            "regiao": regiao.get("nome"),
            "sigla_regiao": regiao.get("sigla"),
            "codigo_regiao": regiao.get("id"),
            "regiao_imediata": regiao_imediata.get("nome"),
            "codigo_regiao_imediata": regiao_imediata.get("id"),
            "regiao_intermediaria": regiao_intermediaria.get("nome"),
            "codigo_regiao_intermediaria": regiao_intermediaria.get("id"),
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(["uf", "municipio"]).reset_index(drop=True)

    return df


In [ ]:
dim_municipio = baixar_municipios_ibge()

dim_municipio.head()


In [ ]:
dim_municipio.to_parquet(TRUSTED_DIR / "dim_municipio.parquet", index=False)
# dim_municipio.to_csv(TRUSTED_DIR / "dim_municipio.csv", index=False, encoding="utf-8")


In [ ]:
print(dim_municipio.shape)
print(dim_municipio["codigo_ibge"].nunique())
print(dim_municipio.head())


## Função consultar SIDRA

In [ ]:
SIDRA_BASE_URL = "https://apisidra.ibge.gov.br/values"


def sidra_request(api_path: str, sleep_seconds: float = 0.5) -> pd.DataFrame:
    """
    Consulta a API SIDRA usando o caminho depois de /values.

    Exemplo de api_path:
        /t/5938/n6/all/v/37/p/last%201/d/v37%202
    """
    url = SIDRA_BASE_URL + api_path

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    data = response.json()

    if not data or len(data) <= 1:
        return pd.DataFrame()

    header = data[0]
    rows = data[1:]

    df = pd.DataFrame(rows)

    # Renomeia colunas usando os nomes descritivos do SIDRA
    rename_map = {col: header.get(col, col) for col in df.columns}
    df = df.rename(columns=rename_map)

    time.sleep(sleep_seconds)

    return df


In [ ]:
def find_column(
    df: pd.DataFrame,
    keywords: List[str],
    exclude_keywords: Optional[List[str]] = None,
) -> Optional[str]:
    """
    Return the first column whose name contains all keywords (ASCII-normalized).
    Skip columns whose name contains any exclude_keywords (e.g. avoid "Sexo (Código)" when matching "sexo").
    """
    cols = list(df.columns)
    exclude_keywords = exclude_keywords or []

    for col in cols:
        col_norm = unidecode(str(col)).lower()
        if any(unidecode(ex).lower() in col_norm for ex in exclude_keywords):
            continue
        if all(unidecode(k).lower() in col_norm for k in keywords):
            return col

    return None


def parse_sidra_valor(series: pd.Series) -> pd.Series:
    """
    Converte a coluna Valor do SIDRA para número.
    Trata símbolos especiais como '...', '-', 'X' etc.
    """
    return (
        series.astype(str)
        .str.strip()
        .replace({
            "...": None,
            "-": None,
            "X": None,
            "x": None,
            "": None,
            "nan": None,
            "None": None,
        })
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )


In [ ]:
def padronizar_sidra_municipal(
    df: pd.DataFrame,
    nome_variavel: str
) -> pd.DataFrame:
    """
    Padroniza o retorno SIDRA em formato cidade-período-valor.
    """
    if df.empty:
        return df

    col_mun_cod = find_column(df, ["municipio", "codigo"])
    col_mun_nome = find_column(df, ["municipio"], exclude_keywords=["codigo"])
    col_ano = find_column(df, ["ano"]) or find_column(df, ["periodo"])
    col_valor = find_column(df, ["valor"])

    if col_mun_cod is None:
        raise ValueError(f"Municipality code column not found. Columns: {df.columns.tolist()}")

    if col_valor is None:
        raise ValueError(f"Value column not found. Columns: {df.columns.tolist()}")

    out = pd.DataFrame({
        "codigo_ibge": df[col_mun_cod].astype(str).str.zfill(7),
        "ano": df[col_ano].astype(str) if col_ano is not None else None,
        nome_variavel: parse_sidra_valor(df[col_valor])
    })

    if col_mun_nome is not None:
        out["municipio_sidra"] = df[col_mun_nome].astype(str)

    out = out.drop_duplicates()

    return out


## População estimada municipal — SIDRA tabela 6579

In [ ]:
def baixar_populacao_estimada_sidra(periodo: str = "last%201") -> pd.DataFrame:
    """
    Baixa população residente estimada municipal pela tabela SIDRA 6579.

    periodo:
        "last%201" -> último período disponível
        "2021"     -> ano específico
        "all"      -> todos os períodos disponíveis
    """
    api_path = f"/t/6579/n6/all/v/9324/p/{periodo}/d/v9324%200"

    df_raw = sidra_request(api_path)
    df = padronizar_sidra_municipal(df_raw, "populacao_estimada")

    return df


In [ ]:
pop_estimada = baixar_populacao_estimada_sidra(periodo="last%205")

pop_estimada.head()


In [ ]:
pop_estimada.to_parquet(TRUSTED_DIR / "sidra_populacao_estimada.parquet", index=False)


In [ ]:
print(pop_estimada.shape)
print(pop_estimada["codigo_ibge"].nunique())
print(pop_estimada["populacao_estimada"].describe())


## População do Censo 2022 - SIDRA tabela 608

In [ ]:
def baixar_populacao_censo_2010_sidra() -> pd.DataFrame:
    """
    Baixa população residente municipal do Censo 2010.

    Tenta, em ordem:
        1. Tabela 608, variável 93, situação do domicílio Total e sexo Total;
        2. Tabela 202, variável 93, situação do domicílio Total e sexo Total;
        3. Variações com classificações invertidas.

    Retorna:
        codigo_ibge
        ano
        populacao_censo_2010
        municipio_sidra
    """

    tentativas = [
        # Tabela 608 - População residente, por situação do domicílio e sexo
        # c1 e c2 normalmente são situação do domicílio e sexo.
        "/t/608/n6/all/v/93/p/2010/c1/0/c2/0",
        "/t/608/n6/all/v/93/p/2010/c2/0/c1/0",

        # Tabela 202 - População residente, por sexo e situação do domicílio
        "/t/202/n6/all/v/93/p/2010/c1/0/c2/0",
        "/t/202/n6/all/v/93/p/2010/c2/0/c1/0",

        # Fallback: baixa todas as categorias e filtra depois
        "/t/608/n6/all/v/93/p/2010/c1/all/c2/all",
        "/t/202/n6/all/v/93/p/2010/c1/all/c2/all",
    ]

    erros = []

    df_raw = None
    api_path_usado = None

    for api_path in tentativas:
        try:
            print(f"Tentando: {api_path}")
            tmp = sidra_request(api_path)

            if tmp is not None and not tmp.empty:
                df_raw = tmp.copy()
                api_path_usado = api_path
                print(f"Sucesso com: {api_path}")
                break

        except Exception as e:
            erros.append((api_path, str(e)))
            print(f"Falhou: {api_path}")
            continue

    if df_raw is None or df_raw.empty:
        print("Todas as tentativas falharam.")
        for path, erro in erros:
            print("\nPATH:", path)
            print("ERRO:", erro[:500])
        return pd.DataFrame()

    # Se a tentativa trouxe todas as categorias, filtra Total.
    # Mantém compatibilidade com suas funções find_column e padronizar_sidra_municipal.
    col_sexo = find_column(
        df_raw,
        ["sexo"],
        exclude_keywords=["codigo"]
    )

    if col_sexo is not None:
        valores_sexo = (
            df_raw[col_sexo]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        if "TOTAL" in set(valores_sexo):
            df_raw = df_raw[valores_sexo.eq("TOTAL")].copy()

    col_situacao = find_column(
        df_raw,
        ["situacao", "domicilio"],
        exclude_keywords=["codigo"]
    )

    if col_situacao is not None:
        valores_situacao = (
            df_raw[col_situacao]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        if "TOTAL" in set(valores_situacao):
            df_raw = df_raw[valores_situacao.eq("TOTAL")].copy()

    df = padronizar_sidra_municipal(
        df_raw,
        "populacao_censo_2010"
    )

    df["ano"] = "2010"
    df["fonte_sidra_populacao_censo_2010"] = api_path_usado

    cols_ordem = [
        "codigo_ibge",
        "ano",
        "populacao_censo_2010",
        "municipio_sidra",
        "fonte_sidra_populacao_censo_2010",
    ]

    cols_ordem = [col for col in cols_ordem if col in df.columns]

    df = df[
        cols_ordem + [col for col in df.columns if col not in cols_ordem]
    ]

    df = (
        df
        .drop_duplicates(["codigo_ibge", "ano"])
        .sort_values("codigo_ibge")
        .reset_index(drop=True)
    )

    return df


In [ ]:
populacao_censo_2010 = baixar_populacao_censo_2010_sidra()

populacao_censo_2010.head()


In [ ]:
populacao_censo_2010 = populacao_censo_2010.drop(columns=["fonte_sidra_populacao_censo_2010"])
populacao_censo_2010


In [ ]:
print(populacao_censo_2010.shape)
if not populacao_censo_2010.empty and "codigo_ibge" in populacao_censo_2010.columns:
    print(populacao_censo_2010["codigo_ibge"].nunique())
if not populacao_censo_2010.empty and "populacao_censo_2010" in populacao_censo_2010.columns:
    print(populacao_censo_2010["populacao_censo_2010"].sum())


In [ ]:
populacao_censo_2010.to_parquet(TRUSTED_DIR / "sidra_populacao_censo_2010.parquet", index=False)


## População do Censo 2022 — SIDRA tabela 9514

In [ ]:
def baixar_populacao_censo_2022_sidra() -> pd.DataFrame:
    """
    Baixa população residente do Censo 2022 pela tabela SIDRA 9514.

    Observação:
    A tabela 9514 possui classificações de sexo, idade e forma de declaração.
    Aqui usamos idade total e forma de declaração total.
    """
    api_path = (
        "/t/9514"
        "/n6/all"
        "/v/allxp"
        "/p/all"
        "/c2/all"
        "/c287/100362"
        "/c286/113635"
    )

    df_raw = sidra_request(api_path)

    if df_raw.empty:
        return df_raw

    # Use label column "Sexo", not "Sexo (Código)" (codes never equal "TOTAL")
    col_sexo = find_column(df_raw, ["sexo"], exclude_keywords=["codigo"])
    if col_sexo is not None:
        df_raw = df_raw[df_raw[col_sexo].astype(str).str.strip().str.upper() == "TOTAL"].copy()

    df = padronizar_sidra_municipal(df_raw, "populacao_censo_2022")
    df["ano"] = "2022"

    return df


In [ ]:
pop_censo_2022 = baixar_populacao_censo_2022_sidra()

pop_censo_2022.head()


In [ ]:
pop_censo_2022.to_parquet(TRUSTED_DIR / "sidra_populacao_censo_2022.parquet", index=False)


In [ ]:
print(pop_censo_2022.shape)
if not pop_censo_2022.empty and "codigo_ibge" in pop_censo_2022.columns:
    print(pop_censo_2022["codigo_ibge"].nunique())
if not pop_censo_2022.empty and "populacao_censo_2022" in pop_censo_2022.columns:
    print(pop_censo_2022["populacao_censo_2022"].sum())


## Juntar Censos: 2010 e 2022

In [ ]:
pop_censos = pop_censo_2022[
    [
        "codigo_ibge",
        "ano",
        "populacao_censo_2022",
    ]
].merge(
    populacao_censo_2010[
        [
            "codigo_ibge",
            "populacao_censo_2010",
        ]
    ],
    on="codigo_ibge",
    how="outer"
)

pop_censos.head()


In [ ]:
pop_censos["crescimento_populacao_censo_2010_2022"] = (
    pop_censos["populacao_censo_2022"]
    / pop_censos["populacao_censo_2010"].replace(0, np.nan)
    - 1
)

pop_censos["cagr_populacao_censo_2010_2022"] = np.where(
    (pop_censos["populacao_censo_2010"] > 0)
    & (pop_censos["populacao_censo_2022"] > 0),
    (
        pop_censos["populacao_censo_2022"]
        / pop_censos["populacao_censo_2010"]
    ) ** (1 / 12) - 1,
    np.nan
)


In [ ]:
pop_censos[
    [
        "codigo_ibge",
        "populacao_censo_2010",
        "populacao_censo_2022",
        "crescimento_populacao_censo_2010_2022",
        "cagr_populacao_censo_2010_2022",
    ]
].head()


<!-- ## PIB Municipal — SIDRA tabela 5938 -->

In [ ]:
pop_censos.to_parquet(
    TRUSTED_DIR / "populacao_censos_2010_2022_municipio.parquet",
    index=False
)


## PIB Municipal

In [ ]:
def baixar_pib_municipal_total(periodo: str = "last%201") -> pd.DataFrame:
    """
    Baixa PIB municipal total a preços correntes pela SIDRA 5938.

    Variável 37:
        Produto Interno Bruto a preços correntes.
    """
    api_path = f"/t/5938/n6/all/v/37/p/{periodo}/d/v37%202"

    df_raw = sidra_request(api_path)
    df = padronizar_sidra_municipal(df_raw, "pib_corrente")

    return df


In [ ]:
pib_total = baixar_pib_municipal_total(periodo="last%205")

pib_total.head()


In [ ]:
pib_total.to_parquet(TRUSTED_DIR / "sidra_pib_total.parquet", index=False)


## Médias Móveis - PIB

In [ ]:
def criar_variaveis_ano_a_ano(
    df: pd.DataFrame,
    col_id: str = "codigo_ibge",
    col_ano: str = "ano",
    col_valor: str = "pib_corrente",
    prefixo: str = "pib_corrente"
) -> pd.DataFrame:
    """
    Cria variáveis ano contra ano para uma base em painel município x ano.

    Espera uma base com:
        codigo_ibge | ano | variável numérica

    Exemplo:
        codigo_ibge | ano | pib_corrente
    """

    out = df.copy()

    # Padronização básica
    out[col_id] = (
        out[col_id]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    out[col_ano] = pd.to_numeric(out[col_ano], errors="coerce").astype("Int64")
    out[col_valor] = pd.to_numeric(out[col_valor], errors="coerce")

    # Ordenação correta para cálculo temporal
    out = out.sort_values([col_id, col_ano]).reset_index(drop=True)

    # Validação de duplicidade
    duplicadas = out.duplicated([col_id, col_ano]).sum()

    if duplicadas > 0:
        print(f"Atenção: foram encontradas {duplicadas} duplicidades por {col_id}-{col_ano}.")
        print("Consolidando por média da variável numérica.")

        cols_first = [
            col for col in out.columns
            if col not in [col_valor]
        ]

        agg_dict = {col: "first" for col in cols_first if col not in [col_id, col_ano]}
        agg_dict[col_valor] = "mean"

        out = (
            out
            .groupby([col_id, col_ano], as_index=False)
            .agg(agg_dict)
            .sort_values([col_id, col_ano])
            .reset_index(drop=True)
        )

    g = out.groupby(col_id, group_keys=False)

    # Lags
    out[f"{prefixo}_lag_1y"] = g[col_valor].shift(1)
    out[f"{prefixo}_lag_2y"] = g[col_valor].shift(2)
    out[f"{prefixo}_lag_3y"] = g[col_valor].shift(3)
    # out[f"{prefixo}_lag_4y"] = g[col_valor].shift(4)

    # Variação absoluta
    out[f"{prefixo}_var_abs_1y"] = (
        out[col_valor] - out[f"{prefixo}_lag_1y"]
    )

    out[f"{prefixo}_var_abs_2y"] = (
        out[col_valor] - out[f"{prefixo}_lag_2y"]
    )

    out[f"{prefixo}_var_abs_3y"] = (
        out[col_valor] - out[f"{prefixo}_lag_3y"]
    )

    # Crescimento percentual ano contra ano
    out[f"{prefixo}_crescimento_1y"] = (
        out[col_valor] / out[f"{prefixo}_lag_1y"] - 1
    )

    out[f"{prefixo}_crescimento_2y_acumulado"] = (
        out[col_valor] / out[f"{prefixo}_lag_2y"] - 1
    )

    out[f"{prefixo}_crescimento_3y_acumulado"] = (
        out[col_valor] / out[f"{prefixo}_lag_3y"] - 1
    )

    # CAGR: crescimento médio anual composto
    out[f"{prefixo}_cagr_2y"] = np.where(
        (out[col_valor] > 0) & (out[f"{prefixo}_lag_2y"] > 0),
        (out[col_valor] / out[f"{prefixo}_lag_2y"]) ** (1 / 2) - 1,
        np.nan
    )

    out[f"{prefixo}_cagr_3y"] = np.where(
        (out[col_valor] > 0) & (out[f"{prefixo}_lag_3y"] > 0),
        (out[col_valor] / out[f"{prefixo}_lag_3y"]) ** (1 / 3) - 1,
        np.nan
    )

    # Log e variação logarítmica
    out[f"log_{prefixo}"] = np.where(
        out[col_valor] > 0,
        np.log(out[col_valor]),
        np.nan
    )

    out[f"{prefixo}_delta_log_1y"] = (
        out[f"log_{prefixo}"] - g[f"log_{prefixo}"].shift(1)
    )

    # Aceleração: crescimento atual menos crescimento anterior
    out[f"{prefixo}_crescimento_1y_lag_1y"] = (
        g[f"{prefixo}_crescimento_1y"].shift(1)
    )

    out[f"{prefixo}_aceleracao_crescimento_1y"] = (
        out[f"{prefixo}_crescimento_1y"]
        - out[f"{prefixo}_crescimento_1y_lag_1y"]
    )

    # Médias móveis
    out[f"{prefixo}_media_movel_3y"] = (
        g[col_valor]
        .rolling(window=3, min_periods=2)
        .mean()
        .reset_index(level=0, drop=True)
    )

    out[f"{prefixo}_desvio_movel_3y"] = (
        g[col_valor]
        .rolling(window=3, min_periods=2)
        .std()
        .reset_index(level=0, drop=True)
    )

    out[f"{prefixo}_coef_var_movel_3y"] = (
        out[f"{prefixo}_desvio_movel_3y"]
        / out[f"{prefixo}_media_movel_3y"].replace(0, np.nan)
    )

    # PIB relativo ao primeiro ano disponível do município
    primeiro_valor = g[col_valor].transform("first")

    out[f"{prefixo}_indice_base_primeiro_ano"] = (
        out[col_valor] / primeiro_valor * 100
    )

    out[f"{prefixo}_crescimento_acumulado_primeiro_ano"] = (
        out[col_valor] / primeiro_valor - 1
    )

    # Ranking e percentil dentro de cada ano
    out[f"{prefixo}_rank_ano"] = (
        out.groupby(col_ano)[col_valor]
        .rank(method="min", ascending=False)
    )

    out[f"{prefixo}_percentil_ano"] = (
        out.groupby(col_ano)[col_valor]
        .rank(pct=True, ascending=True)
    )

    # Participação no PIB total dos municípios daquele ano
    total_ano = out.groupby(col_ano)[col_valor].transform("sum")

    out[f"{prefixo}_share_total_ano"] = (
        out[col_valor] / total_ano.replace(0, np.nan)
    )

    return out


In [ ]:
pib_total_var = criar_variaveis_ano_a_ano(
    df=pib_total,
    col_id="codigo_ibge",
    col_ano="ano",
    col_valor="pib_corrente",
    prefixo="pib_corrente"
)

pib_total_var.head(10)


In [ ]:
pib_total_var[pib_total_var["ano"] == 2023]


## Salvar

In [ ]:
pib_total_var[pib_total_var["ano"] == 2023].to_parquet(TRUSTED_DIR / "sidra_pib_anoxano.parquet", index=False)


## PIB Setorial

In [ ]:
PIB_SETORIAL_VARS = {
    "vab_agropecuaria": 513,
    "vab_industria": 517,
    "vab_servicos": 525,
    "vab_adm_publica": 543,
    "impostos_liquidos": 498,
}


In [ ]:
def baixar_pib_setorial(periodo: str = "2021") -> pd.DataFrame:
    """
    Baixa variáveis setoriais do PIB municipal pela tabela SIDRA 5938.

    Use período específico, por exemplo 2021, pois a edição 2022-2023
    pode não ter abertura setorial tradicional.
    """
    dfs = []

    for nome_variavel, codigo_variavel in PIB_SETORIAL_VARS.items():
        api_path = (
            f"/t/5938"
            f"/n6/all"
            f"/v/{codigo_variavel}"
            f"/p/{periodo}"
            f"/d/v{codigo_variavel}%202"
        )

        df_raw = sidra_request(api_path)
        df = padronizar_sidra_municipal(df_raw, nome_variavel)
        if df.empty or not {"codigo_ibge", "ano"}.issubset(df.columns):
            raise ValueError(
                f"Empty or invalid SIDRA response for variable {codigo_variavel} ({nome_variavel})."
            )

        dfs.append(df[["codigo_ibge", "ano", nome_variavel]])

    base = dfs[0]

    for df in dfs[1:]:
        base = base.merge(df, on=["codigo_ibge", "ano"], how="outer")

    return base


In [ ]:
pib_setorial = baixar_pib_setorial(periodo="last%205")

pib_setorial.head()


In [ ]:
pib_setorial_2021 = pib_setorial[pib_setorial["ano"] <= "2021"]
pib_setorial_2021


In [ ]:
pib_setorial_2021.to_parquet(TRUSTED_DIR / "sidra_pib_setorial.parquet", index=False)


## Ano vs Ano: PIB Setorial

In [ ]:
def criar_variaveis_ano_a_ano_multiplas(
    df: pd.DataFrame,
    col_id: str = "codigo_ibge",
    col_ano: str = "ano",
    cols_valores: Optional[List[str]] = None,
    criar_logs: bool = True,
    criar_ranking_ano: bool = True,
    criar_rolling: bool = True
) -> pd.DataFrame:
    """
    Cria variáveis ano contra ano para múltiplas colunas numéricas em painel município x ano.

    Espera uma base no formato:
        codigo_ibge | ano | vab_agropecuaria | vab_industria | ...

    Retorna a base original acrescida de:
        - lags
        - variações absolutas
        - crescimentos percentuais
        - CAGR
        - delta log
        - aceleração do crescimento
        - médias móveis
        - rankings anuais
        - participação no total anual
    """

    out = df.copy()

    # Padronização das chaves
    out[col_id] = (
        out[col_id]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    out[col_ano] = pd.to_numeric(out[col_ano], errors="coerce").astype("Int64")

    # Identifica automaticamente colunas numéricas, se não forem passadas
    if cols_valores is None:
        cols_valores = []

        for col in out.columns:
            if col in [col_id, col_ano]:
                continue

            serie_num = pd.to_numeric(out[col], errors="coerce")

            if serie_num.notna().mean() > 0.8:
                cols_valores.append(col)

    # Garante conversão numérica
    for col in cols_valores:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # Ordenação temporal
    out = out.sort_values([col_id, col_ano]).reset_index(drop=True)

    # Tratamento de duplicidade por município-ano
    duplicadas = out.duplicated([col_id, col_ano]).sum()

    if duplicadas > 0:
        print(f"Atenção: foram encontradas {duplicadas} duplicidades por {col_id}-{col_ano}.")
        print("Consolidando por média nas colunas numéricas e primeiro valor nas demais.")

        cols_nao_valores = [
            col for col in out.columns
            if col not in cols_valores + [col_id, col_ano]
        ]

        agg_dict = {col: "first" for col in cols_nao_valores}

        for col in cols_valores:
            agg_dict[col] = "mean"

        out = (
            out
            .groupby([col_id, col_ano], as_index=False)
            .agg(agg_dict)
            .sort_values([col_id, col_ano])
            .reset_index(drop=True)
        )

    for col in cols_valores:
        prefixo = col

        grupo_col = out.groupby(col_id)[col]

        # Lags
        out[f"{prefixo}_lag_1y"] = grupo_col.shift(1)
        out[f"{prefixo}_lag_2y"] = grupo_col.shift(2)
        # out[f"{prefixo}_lag_3y"] = grupo_col.shift(3)

        # Variações absolutas
        out[f"{prefixo}_var_abs_1y"] = (
            out[col] - out[f"{prefixo}_lag_1y"]
        )

        out[f"{prefixo}_var_abs_2y"] = (
            out[col] - out[f"{prefixo}_lag_2y"]
        )

        # Crescimentos percentuais
        out[f"{prefixo}_crescimento_1y"] = (
            out[col] / out[f"{prefixo}_lag_1y"].replace(0, np.nan) - 1
        )

        out[f"{prefixo}_crescimento_2y_acumulado"] = (
            out[col] / out[f"{prefixo}_lag_2y"].replace(0, np.nan) - 1
        )

        # CAGR
        out[f"{prefixo}_cagr_2y"] = np.where(
            (out[col] > 0) & (out[f"{prefixo}_lag_2y"] > 0),
            (out[col] / out[f"{prefixo}_lag_2y"]) ** (1 / 2) - 1,
            np.nan
        )

        # Log e delta log
        if criar_logs:
            out[f"log_{prefixo}"] = np.where(
                out[col] > 0,
                np.log(out[col]),
                np.nan
            )

            out[f"{prefixo}_delta_log_1y"] = (
                out[f"log_{prefixo}"]
                - out.groupby(col_id)[f"log_{prefixo}"].shift(1)
            )

        # Aceleração do crescimento
        out[f"{prefixo}_crescimento_1y_lag_1y"] = (
            out.groupby(col_id)[f"{prefixo}_crescimento_1y"].shift(1)
        )

        out[f"{prefixo}_aceleracao_crescimento_1y"] = (
            out[f"{prefixo}_crescimento_1y"]
            - out[f"{prefixo}_crescimento_1y_lag_1y"]
        )

        # Ranking dentro do ano
        if criar_ranking_ano:
            out[f"{prefixo}_rank_ano"] = (
                out.groupby(col_ano)[col]
                .rank(method="min", ascending=False)
            )

            out[f"{prefixo}_percentil_ano"] = (
                out.groupby(col_ano)[col]
                .rank(pct=True, ascending=True)
            )

        # Participação no total do ano
        total_ano = out.groupby(col_ano)[col].transform("sum")

        out[f"{prefixo}_share_total_ano"] = (
            out[col] / total_ano.replace(0, np.nan)
        )

    return out


In [ ]:
cols_pib_setorial = [
    "vab_agropecuaria",
    "vab_industria",
    "vab_servicos",
    "vab_adm_publica",
    "impostos_liquidos",
]

pib_setorial_var = criar_variaveis_ano_a_ano_multiplas(
    df=pib_setorial_2021,
    col_id="codigo_ibge",
    col_ano="ano",
    cols_valores=cols_pib_setorial,
    criar_logs=True,
    criar_ranking_ano=True,
    criar_rolling=False
)


In [ ]:
pib_setorial_var.columns


In [ ]:
pib_setorial_var[pib_setorial_var["ano"] == 2021]


In [ ]:
pib_setorial_var[(pib_setorial_var["codigo_ibge"] == "3509502")&(pib_setorial_var["ano"] == 2021)]


In [ ]:
pib_setorial_var_2021 = pib_setorial_var[pib_setorial_var["ano"] == 2021]
pib_setorial_var_2021.head()


In [ ]:
pib_setorial_var_2021.to_parquet(TRUSTED_DIR / "sidra_pib_setorial_anoxano.parquet", index=False)


## Montar ABT municipal IBGE

In [ ]:
def montar_abt_ibge_painel(
    trusted_dir: Path,
    analytics_dir: Path,
    arquivo_dim_municipio: str = "dim_municipio.parquet",
    arquivo_pib_total: str = "sidra_pib_anoxano.parquet",
    arquivo_pib_setorial: str = "sidra_pib_setorial_anoxano.parquet",
    arquivo_pop_censo: Optional[str] = "populacao_censos_2010_2022_municipio.parquet",
    salvar: bool = True
) -> pd.DataFrame:
    """
    Monta ABT municipal em formato painel cidade-ano usando:
    - dimensão de municípios
    - população estimada dos últimos 5 anos
    - PIB municipal total dos últimos 5 anos
    - PIB setorial dos últimos 5 anos
    - população do Censo 2022, opcional, repetida por município

    A granularidade final é:
        codigo_ibge × ano
    """

    # -----------------------------
    # 1. Leitura das bases
    # -----------------------------
    dim = pd.read_parquet(trusted_dir / arquivo_dim_municipio)
    pop_censo = pd.read_parquet(trusted_dir / arquivo_pop_censo)
    pib_total = pd.read_parquet(trusted_dir / arquivo_pib_total).drop(columns=["ano"])
    pib_setorial = pd.read_parquet(trusted_dir / arquivo_pib_setorial).drop(columns=["ano"])

    # -----------------------------
    # 2. Padronização
    # -----------------------------
    dim = dim.copy()
    dim["codigo_ibge"] = dim["codigo_ibge"].astype(str).str.zfill(7)

    # -----------------------------
    # 3. Construção do painel base
    # -----------------------------
    # A população estimada define o esqueleto cidade-ano.
    # Em tese, deve trazer todos os municípios em todos os anos disponíveis.
    
    painel = pop_censo.copy()

    # -----------------------------
    # 4. Junção com PIB total e PIB setorial
    # -----------------------------
    painel = (
        painel
        .merge(
            pib_total,
            on=["codigo_ibge"],
            how="left"
        )
        .merge(
            pib_setorial,
            on=["codigo_ibge"],
            how="left"
        )
    )

    # -----------------------------
    # 5. Junção com dimensão municipal
    # -----------------------------
    abt = (
        painel
        .merge(
            dim,
            on="codigo_ibge",
            how="left"
        )
    )

    # -----------------------------
    # 7. Organização das colunas
    # -----------------------------
    colunas_identificacao = [
        "codigo_ibge",
        "ano",
        "municipio",
        "uf",
        "nome_uf",
        "regiao",
        "sigla_regiao",
        "regiao_imediata",
        "regiao_intermediaria"
    ]

    colunas_identificacao = [
        col for col in colunas_identificacao if col in abt.columns
    ]

    demais_colunas = [
        col for col in abt.columns if col not in colunas_identificacao
    ]

    abt = abt[colunas_identificacao + demais_colunas]

    # -----------------------------
    # 8. Features derivadas
    # -----------------------------

    # Logs para reduzir assimetria
    if "populacao_censo_2022" in abt.columns:
        abt["log_populacao_censo_2022"] = np.log1p(abt["populacao_censo_2022"])

    if "pib_corrente" in abt.columns:
        abt["log_pib_corrente"] = np.log1p(abt["pib_corrente"])

    # PIB per capita aproximado
    # Atenção: dependendo do retorno SIDRA, pib_corrente pode estar em mil reais.
    # Se estiver em mil reais, usamos * 1000.
    if {"pib_corrente", "populacao_censo_2022"}.issubset(abt.columns):
        abt["pib_per_capita_aprox"] = np.where(
            abt["populacao_censo_2022"] > 0,
            (abt["pib_corrente"] * 1000) / abt["populacao_censo_2022"],
            np.nan
        )

    # Participações setoriais no PIB/VAB
    cols_setoriais = [
        "vab_agropecuaria",
        "vab_industria",
        "vab_servicos",
        "vab_adm_publica"
    ]

    cols_setoriais_existentes = [
        col for col in cols_setoriais if col in abt.columns
    ]

    if len(cols_setoriais_existentes) > 0:
        abt["vab_total_setorial"] = abt[cols_setoriais_existentes].sum(axis=1, min_count=1)

        for col in cols_setoriais_existentes:
            abt[f"share_{col}"] = np.where(
                abt["vab_total_setorial"] > 0,
                abt[col] / abt["vab_total_setorial"],
                np.nan
            )

    # Razão serviços + administração pública
    if {"vab_servicos", "vab_adm_publica"}.issubset(abt.columns):
        abt["vab_servicos_total"] = abt["vab_servicos"] + abt["vab_adm_publica"]

        if "vab_total_setorial" in abt.columns:
            abt["share_servicos_total"] = np.where(
                abt["vab_total_setorial"] > 0,
                abt["vab_servicos_total"] / abt["vab_total_setorial"],
                np.nan
            )

    # Crescimento populacional e econômico ano contra ano
    abt = abt.sort_values(["codigo_ibge", "ano"]).reset_index(drop=True)

    # -----------------------------
    # 9. Salvar
    # -----------------------------
    if salvar:
        analytics_dir.mkdir(parents=True, exist_ok=True)

        abt.to_parquet(
            analytics_dir / "abt_municipios_ibge_painel.parquet",
            index=False
        )

        # abt.to_csv(
        #     analytics_dir / "abt_municipios_ibge_painel.csv",
        #     index=False,
        #     encoding="utf-8"
        # )

    return abt


In [ ]:
abt_ibge = montar_abt_ibge_painel(
    trusted_dir=TRUSTED_DIR,
    analytics_dir=ANALYTICS_DIR
)

abt_ibge.head()


In [ ]:
abt_ibge.columns.to_list()


In [ ]:
abt_ibge[abt_ibge["municipio"] == "Campinas"]


In [ ]:
abt_ibge[abt_ibge["municipio"] == "Paranaguá"]


In [ ]:
abt_ibge.to_parquet(ANALYTICS_DIR / "abt_municipios_ibge_basica.parquet", index=False)


In [ ]:
abt_ibge.columns.to_list()


## Leitura de Dados

In [ ]:
df_ibge_painel = pd.read_parquet(ANALYTICS_DIR / "abt_municipios_ibge_painel.parquet")[
    ["codigo_ibge", 
    "ano", "municipio",
    "populacao_estimada",
    "pib_corrente",
    "vab_agropecuaria",
    "vab_industria",
    "vab_servicos",
    "vab_adm_publica",
    "impostos_liquidos",
    "populacao_censo_2022",
    "populacao_censo_2022",
    "log_populacao_estimada",
    "log_pib_corrente",
    "pib_per_capita_aprox",
    "vab_total_setorial",
    "share_vab_agropecuaria",
    "share_vab_industria",
    "share_vab_servicos",
    "share_vab_adm_publica",
    "vab_servicos_total",
    "share_servicos_total",
    "crescimento_populacao_1y",
    "crescimento_pib_corrente_1y",
    "crescimento_pib_per_capita_1y",
    "crescimento_vab_agropecuaria_1y",
    "crescimento_vab_industria_1y",
    "crescimento_vab_servicos_1y",
    "crescimento_vab_adm_publica_1y",
    ]
]
df_ibge_painel


In [ ]:
df_ibge_painel[df_ibge_painel["codigo_ibge"] == "4118204"][
    ['codigo_ibge', 'ano', 'municipio', 'uf', 'nome_uf', 'regiao',
       'sigla_regiao', 'regiao_imediata', 'regiao_intermediaria','municipio_norm',
       'codigo_uf', 'codigo_regiao', 'codigo_regiao_imediata', 'codigo_regiao_intermediaria']
       ]


In [ ]:
df_ibge_painel[df_ibge_painel["codigo_ibge"] == "4118204"][
    ['codigo_ibge', 'ano', 'populacao_censo_2022', 'log_populacao_estimada', 'log_pib_corrente',
       'pib_per_capita_aprox', 'vab_total_setorial', 'share_vab_agropecuaria',
       'share_vab_industria', 'share_vab_servicos', 'share_vab_adm_publica',
       'vab_servicos_total', 'share_servicos_total',
       'crescimento_populacao_1y', 'crescimento_pib_corrente_1y',]
       ]


In [ ]:
df_ibge_painel[df_ibge_painel["codigo_ibge"] == "4118204"]


In [ ]:
df_ibge_painel.columns
